## Converting to standard form

Convert the following problem to standard form:

<p style="text-align: center;">
$\begin{align}
\underset{x_1,x_2}{\min} \ & x_1 + x_2 & \\
\text{s.t.} \ & 5x_1 - 3x_2 = 7 &\\
& 2x_1 + x_2 \geq 2 &\\
& 1 \leq x_2 \leq 4 & \\
& x_1 \text{ free}
\end{align}$
</p>

### Original model

In [6]:
using JuMP, Clp

m = Model(solver=ClpSolver())

@variable(m, x1 )
@variable(m, 1 <= x2 <= 4 )
@constraint(m, 5*x1 - 3*x2 == 7 )
@constraint(m, 2*x1 + x2 >= 2 )
@objective(m, Min, x1 + x2 )

status = solve(m)

println(m)
println(status)
println()
println("x1 = ", getvalue(x1) )
println("x2 = ", getvalue(x2) )
println("objective = ", getobjectivevalue(m) )

Min x1 + x2
Subject to
 5 x1 - 3 x2 = 7
 2 x1 + x2 ≥ 2
 x1
 1 ≤ x2 ≤ 4

Optimal

x1 = 2.0
x2 = 1.0
objective = 3.0


### Standard Form

When we convert it, the model should look like this:
<p style="text-align: center;">
$\begin{align}
\underset{x}{\max} \ & c^Tx & \\
\text{s.t.} \ & Ax \leq b &\\
& x \geq 0 &
\end{align}$
</p>

* We bound $x_1$ by replacing it with $x_1 = u - v, \ u, v \geq 0$:

<p style="text-align: center;">
$\begin{align}
\underset{u,v,x_2}{\min} \ & (u-v) + x_2 & \\
\text{s.t.} \ & 5(u-v) - 3x_2 = 7 &\\
& 2(u-v) + x_2 \geq 2 &\\
& 1 \leq x_2 \leq 4 & \\
& u, v \geq 0&
\end{align}$
</p>

* We shift the bounds on $x_2$ by using $w = x_2 - 1$:

<p style="text-align: center;">
$\begin{align}
\underset{u,v,w}{\min} \ & (u-v) + (w+1) & \\
\text{s.t.} \ & 5(u-v) - 3(w+1) = 7 &\\
& 2(u-v) + (w+1) \geq 2 &\\
& (w+1) \leq 4 & \\
& u, v, w \geq 0&
\end{align}$
</p>

* Rearrange so all constants are on the right-hand-side:

<p style="text-align: center;">
$\begin{align}
\underset{u,v,w}{\min} \ & u - v + w + 1 & \\
\text{s.t.} \ & 5u- 5v - 3w = 10 &\\
& 2u - 2v + w \geq 1 &\\
& w \leq 3 & \\
& u, v, w \geq 0&
\end{align}$
</p>

* We turn the min into a max:
<p style="text-align: center;">
$\begin{align}
\underset{u,v,w}{-\max}\ & -u + v - w - 1 & \\
\text{s.t.} \ & 5u- 5v - 3w = 10 &\\
& 2u - 2v + w \geq 1 &\\
& w \leq 3 & \\
& u, v, w \geq 0&
\end{align}$
</p>

* Flip $\geq$ inequality and replace equality with two inequalities:
<p style="text-align: center;">
$\begin{align}
\underset{u,v,w}{-\max}\ & -u + v - w - 1 & \\
\text{s.t.} \ & 5u- 5v - 3w \leq 10 &\\
& 5u- 5v - 3w \geq 10 &\\
& -2u + 2v - w \leq -1 &\\
& w \leq 3 & \\
& u, v, w \geq 0&
\end{align}$
</p>

* Finally, flip the newly created $\geq$ inequality and we're done!
<p style="text-align: center;">
$\begin{align}
\underset{u,v,w}{-\max}\ & -u + v - w - 1 & \\
\text{s.t.} \ & 5u- 5v - 3w \leq 10 &\\
& -5u + 5v + 3w \leq -10 &\\
& -2u + 2v - w \leq -1 &\\
& w \leq 3 & \\
& u, v, w \geq 0&
\end{align}$
</p>

In [11]:
using JuMP, Clp

m = Model(solver=ClpSolver())
@variable(m, u >= 0 )
@variable(m, v >= 0 )
@variable(m, w >= 0 )
@constraint(m, 5*u - 5*v - 3*w <= 10 )
@constraint(m, -5*u + 5*v + 3*w <= -10 )
@constraint(m, -2*u + 2*v - w <= -1 )
@constraint(m, w <= 3 )
@objective(m, Max, -u + v - w - 1 )

status = solve(m)

println(m)
println(status)
println()
println("x1 = ", getvalue(u-v) )
println("x2 = ", getvalue(w+1) )
println("objective = ", -getobjectivevalue(m) )

Max -u + v - w - 1
Subject to
 5 u - 5 v - 3 w ≤ 10
 -5 u + 5 v + 3 w ≤ -10
 -2 u + 2 v - w ≤ -1
 w ≤ 3
 u ≥ 0
 v ≥ 0
 w ≥ 0

Optimal

x1 = 2.0
x2 = 1.0
objective = 3.0


### Compact Standard Form (with matrices)

In [14]:
# we defined the vector x to be: x = [u;  v;  w]
A = [5 -5 -3; -5 5 3; -2 2 -1; 0 0 1 ]
b = [10; -10; -1; 3]
c = [-1; 1; -1]

using JuMP, Clp

m = Model(solver=ClpSolver())
@variable(m, x[1:3] >= 0 )      # specify a vector variable
@constraint(m, A*x .<= b )      # use a dot in front of <= to indicate element-wise (vector) inequalities
@objective(m, Max, dot(c,x) - 1)   # must use dot(c,x) or (c'*x)[1] to return a scalar

status = solve(m)

println(m)
println(status)
println()
println("x1 = ", getvalue(x[1]-x[2]) )
println("x2 = ", getvalue(x[3]+1) )
println("objective = ", -(getobjectivevalue(m)) )

Max -x[1] + x[2] - x[3] - 1
Subject to
 5 x[1] - 5 x[2] - 3 x[3] ≤ 10
 -5 x[1] + 5 x[2] + 3 x[3] ≤ -10
 -2 x[1] + 2 x[2] - x[3] ≤ -1
 x[3] ≤ 3
 x[i] ≥ 0 ∀ i ∈ {1,2,3}

Optimal

x1 = 2.0
x2 = 1.0
objective = 3.0
